In [ ]:
from selenium import webdriver
from selenium . webdriver.common.by import By
from selenium.webdriver.firefox.service import Service as FirefoxService
from webdriver_manager.firefox import GeckoDriverManager

from selenium.webdriver.chrome.options import Options


opts = Options()
opts.add_argument("--headless")          
opts.add_argument("--no-sandbox")        
opts.add_argument("--disable-dev-shm-usage")


In [ ]:
driver = webdriver.Firefox(service=FirefoxService(GeckoDriverManager().install()))
driver.get("https://fbref.com/en/comps/9/Premier-League-Stats")

In [ ]:
players = []

content = driver.find_element(By.ID, "div_results2025-202691_overall")

In [ ]:
buttons = driver.find_elements(By.XPATH, ".//td[@data-stat='team']/a")

In [ ]:
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

wait = WebDriverWait(driver, 10)
league_url = driver.current_url

buttons = driver.find_elements(By.XPATH, "//td[@data-stat='team']/a")
hrefs = [a.get_attribute('href') for a in buttons]

for href in hrefs:
    driver.get(href)
    table = wait.until(EC.presence_of_element_located((By.ID, "all_stats_standard")))
    #result.append(table.get_attribute('outerHTML'))
    results = table.find_elements(By.CSS_SELECTOR, '[data-stat="player"]')
    results = table.find_elements(By.CSS_SELECTOR, '[data-stat="player"]')
    results = table.find_elements(By.CSS_SELECTOR , '[data-stat="player"]')
    for result in results:
     
        players.append(result.text)
    


In [24]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.firefox.service import Service as FirefoxService
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.firefox import GeckoDriverManager
import pandas as pd

driver = webdriver.Firefox(service=FirefoxService(GeckoDriverManager().install()))
wait = WebDriverWait(driver, 10)

try:
    driver.get("https://fbref.com/en/comps/9/Premier-League-Stats")
    team_links = [a.get_attribute("href") for a in driver.find_elements(By.XPATH, "//td[@data-stat='team']/a")][:20]
    teams = [a.text for a in driver.find_elements(By.XPATH, "//td[@data-stat='team']/a")][:20]
    
    all_rows = []
    headers = []

    all_rows_match = []
    headers_match = []
    
    for team_idx, link in enumerate(team_links):
        driver.get(link)
        table = wait.until(EC.presence_of_element_located((By.ID, "stats_standard_9")))
        
        if not headers:
            header_row = table.find_element(By.XPATH, ".//thead//tr[last()]")
            all_headers = [th.get_attribute("data-stat") or th.text.strip() for th in header_row.find_elements(By.TAG_NAME, "th")]
            headers = all_headers[:16]
        
        rows = table.find_elements(By.XPATH, ".//tbody//tr[not(contains(@class, 'thead'))]")
        
        current_team = teams[team_idx]
        
        for row in rows:
            row_data = {}
            cells = row.find_elements(By.TAG_NAME, "th") + row.find_elements(By.TAG_NAME, "td")
            for cell_idx, cell in enumerate(cells[:16]):
                if cell_idx < len(headers):
                    data_stat = cell.get_attribute("data-stat")
                    if data_stat and data_stat not in row_data:
                        row_data[data_stat] = cell.text.strip()
            
            if row_data:
                row_data['team'] = current_team
                all_rows.append(row_data)


        table_match = wait.until(EC.presence_of_element_located((By.ID , 'matchlogs_for')))
        headers_match = table_match.find_elements(By.XPATH , './/thead//tr//th')

        rows_match = table_match.find_elements(By.XPATH, ".//tbody//tr[not(contains(@class, 'thead'))]")
        for row in rows_match:
            row_data = {}
            cells = row.find_elements(By.TAG_NAME, "th") + row.find_elements(By.TAG_NAME, "td")
            for cell_idx, cell in enumerate(cells):
                if cell_idx < len(headers_match):
                    data_stat = cell.get_attribute("data-stat")
                    if data_stat and data_stat not in row_data:
                        row_data[data_stat] = cell.text.strip()
            
            if row_data:
                
                all_rows_match.append(row_data)



    
    df = pd.DataFrame(all_rows)
    column_order = headers + ['team']
    df = df[column_order]
    df.to_csv("premier_league_stats.csv", index=False)

    df_match = pd.DataFrame(all_rows_match)
    

   

except Exception as e:
    print(f"Error: {e}")
finally:
    driver.quit()

In [25]:
df_match

,date,start_time,comp,round,dayofweek,venue,result,goals_for,goals_against,opponent,xg_for,xg_against,possession,attendance,captain,formation,opp_formation,referee,match_report,notes
0,2025-08-17,16:30,Premier League,Matchweek 1,Sun,Away,W,1,0,Manchester Utd,1.3,1.5,39,"73,475",Martin Ødegaard,4-3-3,3-4-3,Simon Hooper,Match Report,
1,2025-08-23,17:30,Premier League,Matchweek 2,Sat,Home,W,5,0,Leeds United,2.7,0.2,67,"60,110",Martin Ødegaard,4-3-3,4-3-3,Jarred Gillett,Match Report,
2,2025-08-31,16:30,Premier League,Matchweek 3,Sun,Away,L,0,1,Liverpool,0.5,0.5,47,"60,455",Gabriel Magalhães,4-3-3,4-2-3-1,Chris Kavanagh,Match Report,
3,2025-09-13,12:30,Premier League,Matchweek 4,Sat,Home,W,3,0,Nott'ham Forest,1.8,0.2,54,"60,167",Martin Ødegaard,4-3-3,4-2-3-1,Darren England,Match Report,
4,2025-09-16,18:45 (17:45),Champions Lg,League phase,Tue,Away,W,2,0,es Athletic Club,1.3,0.3,61,"51,059",Gabriel Magalhães,4-3-3,4-2-3-1,Donatas Rumšas,Match Report,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
867,2026-04-25,15:00,Premier League,Matchweek 34,Sat,Home,,,,Tottenham,,,,,,,,,Head-to-Head,
868,2026-05-02,15:00,Premier League,Matchweek 35,Sat,Home,,,,Sunderland,,,,,,,,,Head-to-Head,
869,2026-05-09,15:00,Premier League,Matchweek 36,Sat,Away,,,,Brighton,,,,,,,,,Head-to-Head,
870,2026-05-17,15:00,Premier League,Matchweek 37,Sun,Home,,,,Fulham,,,,,,,,,Head-to-Head,


In [6]:
df1['team']

0       Arsenal
1       Arsenal
2       Arsenal
3       Arsenal
4       Arsenal
         ...   
1055        NaN
1056        NaN
1057        NaN
1058        NaN
1059        NaN
Name: team, Length: 1060, dtype: object